# Parametric-only baseline

**Goal:** evaluate the generator without retrieval.

This is the baseline system:

```text
question -> generator -> answer
```

We also compute conditional answer perplexity:

```text
PPL(answer | question)
```

This gives us a fair language-modeling metric to compare with RAG:

```text
PPL(answer | question, retrieved_context)
```


In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.config import PROCESSED_DIR, PREDICTIONS_DIR, METRICS_DIR, GENERATOR_MODEL
from src.generation import (
    load_generator, build_parametric_prompt,
    generate_answer, answer_loss_and_perplexity, estimate_model_size_mb
)
from src.metrics import evaluate_qa_predictions, numeric_summary
from src.analysis_utils import save_json, current_process_memory_mb

pd.set_option("display.max_colwidth", 160)

In [4]:
questions_df = pd.read_parquet(PROCESSED_DIR / "questions.parquet")

EVAL_N = min(2000, len(questions_df))
eval_df = questions_df.head(EVAL_N).copy()

print(eval_df.shape)
display(eval_df.head(3))

(2000, 7)


,sample_id,question,answer,type,level,support_titles,gold_doc_ids
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,comparison,hard,"[""Ed Wood"", ""Scott Derrickson""]","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]"
1,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,bridge,hard,"[""Kiss and Tell (1945 film)"", ""Shirley Temple""]","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]"
2,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,bridge,hard,"[""Animorphs"", ""The Hork-Bajir Chronicles""]","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]"


In [5]:
tokenizer, model, device = load_generator(GENERATOR_MODEL)
print("Device:", device)
print("Approx model size MB:", estimate_model_size_mb(model))
print("Process memory MB:", current_process_memory_mb())

Device: mps
Approx model size MB: 1884.58544921875
Process memory MB: 719.171875


In [6]:
rows = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Parametric baseline"):
    prompt = build_parametric_prompt(row["question"])

    pred, gen_latency = generate_answer(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=48,
        temperature=0.0,
    )

    ppl_info = answer_loss_and_perplexity(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        answer=row["answer"],
    )

    rows.append({
        "system": "parametric_only",
        "sample_id": row["sample_id"],
        "question": row["question"],
        "answer": row["answer"],
        "prediction": pred,
        "answer_loss": ppl_info["answer_loss"],
        "answer_ppl": ppl_info["answer_ppl"],
        "answer_n_tokens": ppl_info["answer_n_tokens"],
        "generation_latency_sec": gen_latency,
        "total_latency_sec": gen_latency,
    })

    if (i + 1) % 25 == 0:
        pd.DataFrame(rows).to_csv(PREDICTIONS_DIR / "parametric_predictions_partial.csv", index=False)

pred_df = pd.DataFrame(rows)
pred_df.to_csv(PREDICTIONS_DIR / "parametric_predictions.csv", index=False)

display(pred_df.head())

Parametric baseline:   0%|          | 0/2000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,system,sample_id,question,answer,prediction,answer_loss,answer_ppl,answer_n_tokens,generation_latency_sec,total_latency_sec
0,parametric_only,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"No, they are not from the same nationality.",8.175336,3552.247867,1,2.867844,2.867844
1,parametric_only,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"The woman who played Corliss Archer in the 1984 film ""Kiss and Tell"" is actress Mary Elizabeth Winstead. She portrayed the character of Corliss Archer, a fo...",5.605670,271.964203,3,1.751931,1.751931
2,parametric_only,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,"""The Book Thief"" by Markus Zusak is an example of a science fantasy young adult series that tells its story through first-person narration.",5.001980,148.707283,3,2.350215,2.350215
3,parametric_only,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,"No, they are not located in the same neighborhood.",5.245604,189.730295,1,1.521451,1.521451
4,parametric_only,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City",New York City,1.415315,4.117784,6,1.555111,1.555111


In [ ]:
metrics = {
    "system": "parametric_only",
}

metrics.update(evaluate_qa_predictions(rows))

metrics.update(numeric_summary(pred_df["answer_loss"], "answer_loss"))
metrics.update(numeric_summary(pred_df["answer_ppl"], "answer_ppl"))
metrics.update(numeric_summary(pred_df["generation_latency_sec"], "generation_latency_sec"))
metrics.update(numeric_summary(pred_df["total_latency_sec"], "total_latency_sec"))

metrics["model_name"] = GENERATOR_MODEL
metrics["model_size_mb_estimate"] = estimate_model_size_mb(model)

save_json(metrics, METRICS_DIR / "parametric_metrics.json")

metrics

{'system': 'parametric_only',
 'exact_match': 0.0255,
 'token_f1': 0.08688771437648225,
 'contains_answer': 0.1455,
 'n': 2000,
 'answer_loss_mean': 3.945451378338039,
 'answer_loss_median': 3.389434814453125,
 'answer_loss_p90': 7.731720161437995,
 'answer_loss_p95': 9.029060173034667,
 'answer_loss_max': 18.31439781188965,
 'answer_ppl_mean': 74122.13999444814,
 'answer_ppl_median': 29.649291169064526,
 'answer_ppl_p90': 2279.719996092334,
 'answer_ppl_p95': 8342.264284987306,
 'answer_ppl_max': 89917020.80013104,
 'generation_latency_sec_mean': 2.1109558109149895,
 'generation_latency_sec_median': 2.1126227915001436,
 'generation_latency_sec_p90': 2.1540369214008024,
 'generation_latency_sec_p95': 2.1667485639491133,
 'generation_latency_sec_max': 2.8678435419997186,
 'total_latency_sec_mean': 2.1109558109149895,
 'total_latency_sec_median': 2.1126227915001436,
 'total_latency_sec_p90': 2.1540369214008024,
 'total_latency_sec_p95': 2.1667485639491133,
 'total_latency_sec_max': 2.867

### Results

The parametric-only baseline is intentionally difficult: the model receives only the question and must answer from its internal parameters. On 2,000 HotpotQA examples, it reaches `Exact Match = 0.0255`, `token F1 = 0.0869`, and `contains-answer = 0.1455`. This confirms that a small parametric-only transformer struggles on fact-heavy multi-hop questions without external evidence.

The language-modeling metrics show the same limitation. Mean answer loss is `3.945`, and median answer perplexity is `29.65`. The raw mean perplexity is much larger because perplexity is highly sensitive to a small number of extreme outliers. For this reason, the final comparison should use mean answer loss and median/p90 perplexity, not mean perplexity alone.

The average total latency is `2.11` seconds per question on the current MPS setup. This gives the latency baseline for later RAG systems: any retrieval-augmented method should be judged not only by answer quality, but also by the additional latency it introduces.